# Databricks Notebook: 03_create_economic_data.ipynb

# Este notebook gera dados econômicos simulados e os salva como uma tabela Delta no DBFS.

In [0]:
import logging
from typing import Dict, List, Optional, Any
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)
import random

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

class EconomicDataError(Exception):
    """Custom exception for economic data generation errors."""
    pass

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a SparkSession with Delta Lake support.

    Args:
        app_name (str): The name of the Spark application.

    Returns:
        SparkSession: The configured SparkSession.
    """
    logger.info(f"Creating SparkSession for application: {app_name}")
    
    try:
        spark = SparkSession.builder \
            .appName(app_name) \
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
            .getOrCreate()
        logger.info("SparkSession created successfully.")
        return spark
    except Exception as e:
        logger.error(f"Failed to create SparkSession: {e}")
        raise EconomicDataError(f"SparkSession creation failed: {e}")

def generate_economic_data(spark: SparkSession, num_countries: int = 250) -> DataFrame:
    """
    Generates simulated economic data for a given number of countries.

    Args:
        spark (SparkSession): The active SparkSession.
        num_countries (int): The number of simulated countries to generate data for.

    Returns:
        DataFrame: A Spark DataFrame containing simulated economic data.
    
    Raises:
        EconomicDataError: If data generation fails.
    """
    logger.info(f"Generating simulated economic data for {num_countries} countries.")
    
    try:
        economic_data = [
        ("USA", 65000, 0.926), # Estados Unidos
        ("CHN", 10500, 0.761), # China
        ("JPN", 40000, 0.919), # Japão
        ("DEU", 46000, 0.947), # Alemanha
        ("IND", 2100, 0.645),  # Índia
        ("GBR", 42000, 0.932), # Reino Unido
        ("FRA", 41000, 0.901), # França
        ("BRA", 7500, 0.765),  # Brasil
        ("ITA", 34000, 0.892), # Itália
        ("CAN", 46000, 0.929), # Canadá
        ("KOR", 31000, 0.916), # Coreia do Sul
        ("RUS", 11500, 0.824), # Rússia
        ("AUS", 55000, 0.944), # Austrália
        ("ESP", 29000, 0.904), # Espanha
        ("MEX", 9900, 0.779),  # México
        ("IDN", 4100, 0.718),  # Indonésia
        ("NGA", 2200, 0.539),  # Nigéria (Exemplo de país que pode estar no df_final mas não aqui)
        ("XYZ", 5000, 0.600)   # País fictício que não existe no df_final
        ]

        economic_columns = ["cca3", "gdp_per_capita", "hdi"]
        df_economic = spark.createDataFrame(economic_data, economic_columns)
        logger.info(f"Successfully generated economic data for {num_countries} countries.")
        return df_economic
        
    except Exception as e:
        logger.error(f"Error generating economic data: {e}")
        raise EconomicDataError(f"Failed to generate economic data: {e}")

def validate_economic_data(df: DataFrame) -> bool:
    """
    Validates the economic data DataFrame.
    
    Args:
        df (DataFrame): The DataFrame to validate.
        
    Returns:
        bool: True if validation passes.
        
    Raises:
        EconomicDataError: If validation fails.
    """
    try:
        # Check if DataFrame is not empty
        count = df.count()
        if count == 0:
            raise EconomicDataError("DataFrame is empty")
        
        # Check for required columns
        required_columns = {"cca3", "gdp_per_capita", "hdi"}
        actual_columns = set(df.columns)
        if not required_columns.issubset(actual_columns):
            missing = required_columns - actual_columns
            raise EconomicDataError(f"Missing required columns: {missing}")
        
        # Check for null values
        null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
        if any(null_counts):
            raise EconomicDataError("DataFrame contains null values")
        
        logger.info(f"Data validation passed for {count} records.")
        return True
        
    except Exception as e:
        logger.error(f"Data validation failed: {e}")
        raise EconomicDataError(f"Data validation error: {e}")

def save_economic_data(df: DataFrame, output_path: str) -> None:
    """
    Saves the economic data to a Delta Lake table.

    Args:
        df (DataFrame): The DataFrame containing economic data.
        output_path (str): The path to save the Delta table.
        
    Raises:
        EconomicDataError: If saving fails.
    """
    logger.info(f"Writing economic data to Delta Lake at {output_path}")
    
    try:
        # Validate data before saving
        validate_economic_data(df)
        
        # Save to Delta Lake
        df.write.format("delta").mode("overwrite").save(output_path)
        df.write.mode("overwrite").saveAsTable("economic_data")
        
        # Verify the write was successful
        verification_df = spark.read.format("delta").load(output_path)
        saved_count = verification_df.count()
        original_count = df.count()
        
        if saved_count != original_count:
            raise EconomicDataError(f"Row count mismatch: original={original_count}, saved={saved_count}")
        
        logger.info(f"Economic data successfully saved to Delta Lake. Records: {saved_count}")
        
    except Exception as e:
        logger.error(f"Error saving economic data to Delta Lake: {e}")
        raise EconomicDataError(f"Failed to save economic data: {e}")

def main():
    """Main execution function."""
    spark = None
    try:
        # Initialize Spark session
        spark = create_spark_session("EconomicDataGeneration")
        
        # Configuration
        output_path = "/Volumes/workspace/default/data/simulated/economic_data"
        num_countries = 250  # Can be parameterized via widget or job parameter
        
        # Generate economic data
        economic_df = generate_economic_data(spark, num_countries)
        
        # Save to Delta Lake
        save_economic_data(economic_df, output_path)
        
        # Show sample data for verification
        logger.info("Sample of generated data:")
        economic_df.show(10)
        
        # Return output path for orchestrator notebook
        try:
            dbutils.notebook.exit(output_path)
        except NameError:
            # dbutils might not be available in all environments
            logger.info(f"Output path: {output_path}")
            return output_path
            
    except EconomicDataError as e:
        logger.error(f"Economic data processing error: {e}")
        raise
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        raise EconomicDataError(f"Unexpected error in main execution: {e}")
    finally:
        if spark:
            spark.stop()
            logger.info("SparkSession stopped.")

# Execute main function
if __name__ == "__main__":
    main()